## 去噪过程与采样生成

去噪的过程（即采样生成）本质上就是把前向扩散的 SDE 逆转过来，从一个纯噪声状态 $t=1$ 出发，一步步“走”回 $t=0$ 的真实数据状态。去噪主要有两种主流路径：**基于随机微分方程（SDE）的采样和基于常微分方程（ODE）**的采样。

---

### 1. 核心理论：反向 SDE (Reverse SDE)

根据随机分析理论，任何一个前向扩散 SDE 都有一个对应的反向 SDE。如果前向是：
$$
d\mathbf{x} = \mathbf{f}(\mathbf{x}, t)dt + g(t)d\mathbf{w}
$$

那么从 $T$ 到 $0$ 的反向过程为：
$$
d\mathbf{x} = \left[ \mathbf{f}(\mathbf{x}, t) - g(t)^2 \nabla_{\mathbf{x}} \log p_t(\mathbf{x}) \right] dt + g(t)d\bar{\mathbf{w}}
$$

- $\nabla_{\mathbf{x}} \log p_t(\mathbf{x})$：这就是我们模型学到的分数函数（Score Function）。
- $d\bar{\mathbf{w}}$：这是一个反向的标准布朗运动（从未来向过去震荡）。

---

### 2. 具体的去噪步骤（以迭代方式理解）

在实际操作中，我们无法进行连续积分，所以需要将过程离散化。以下是去噪的通用流程：

#### 第一步：初始化
从标准高斯分布中随机抽取一个纯噪声：
$$
\mathbf{x}_T \sim \mathcal{N}(0, \mathbf{I})
$$

#### 第二步：循环迭代（从 $t=T$ 到 $t=1$）
在每一个时间步，模型执行以下动作：

1. **预测噪声/分数**：将当前的 $\mathbf{x}_t$ 和时间 $t$ 喂给模型，得到 $\boldsymbol{\epsilon}_{\theta}(\mathbf{x}_t, t)$。
2. **计算“确定性”位移**：根据漂移项 $\mathbf{f}$ 和模型预测的分数，算出 $\mathbf{x}$ 应该向均值方向移动多少。
3. **注入“随机”扰动**：为了保持生成的多样性，在每一步走完后，再根据 $g(t)$ 注入一丁点新的随机噪声。

注：这种“边去噪边加噪”的方式被称为 Langevin Dynamics（朗之万动力学），它能帮助样本停留在高密度区域。

---

### 3. VP 框架下的去噪（DDPM 采样器）

在经典的 DDPM (VP SDE) 中，每一退步的公式如下：
$$
\mathbf{x}_{t-1} = \frac{1}{\sqrt{\alpha_t}} \left( \mathbf{x}_t - \frac{1-\alpha_t}{\sqrt{1-\bar{\alpha}_t}} \boldsymbol{\epsilon}_{\theta}(\mathbf{x}_t, t) \right) + \sigma_t \mathbf{z}
$$

其中 $\mathbf{z} \sim \mathcal{N}(0, \mathbf{I})$ 是该步注入的随机扰动。

直观理解：
- 第一部分是利用模型预测的结果把 $\mathbf{x}_t$ 往 $\mathbf{x}_0$ 的方向推；
- 第二部分是增加一点随机性，防止模型生成的图像太死板。

---

### 4. 另一种选择：概率流 ODE (Probability Flow ODE)

除了带随机性的 SDE，Song Yang 等人证明了存在一个确定的 ODE，其演化轨迹产生的边缘分布与 SDE 完全一致：
$$
d\mathbf{x} = \left[ \mathbf{f}(\mathbf{x}, t) - \frac{1}{2}g(t)^2 \nabla_{\mathbf{x}} \log p_t(\mathbf{x}) \right] dt
$$

#### 特点：
- **没有 $d\bar{\mathbf{w}}$ 项**：意味着只要 $\mathbf{x}_T$ 确定了，生成的图像就唯一确定了。
- **优点**：
  - **可逆性**：图像和噪声可以一一对应。
  - **加速采样**：可以使用成熟的 ODE 求解器（如 RK45 或常见的 DPMSolver），大大减少生成所需的步数（从 1000 步减到 20-50 步）。

## PC 采样器：Predictor-Corrector 采样算法

PC 采样器的任务是求解反向 SDE。由于离散化求解必然带来误差，PC 采样器通过“两步走”的策略来保证样本始终留在高概率密度区域。

---

### 1. Predictor (预测器)：时间步进

Predictor 的作用是执行跨越时间的步进。它负责将样本从 $t$ 时刻推向 $t-\Delta t$ 时刻。

#### 核心方程：
通常采用 Euler-Maruyama 离散化方案。对于反向 SDE：
$$
d\mathbf{x} = [\mathbf{f}(\mathbf{x}, t) - g(t)^2 \nabla_{\mathbf{x}} \log p_t(\mathbf{x})] dt + g(t)d\bar{\mathbf{w}}
$$

离散化迭代：
$$
\mathbf{x}_{t-\Delta t} = \mathbf{x}_t + \underbrace{[\mathbf{f}(\mathbf{x}_t, t) - g(t)^2 \mathbf{s}_\theta(\mathbf{x}_t, t)]}_{\text{确定性漂移}} (-\Delta t) + \underbrace{g(t)\sqrt{\Delta t} \mathbf{z}}_{\text{随机扩散}}
$$

- **目的**：改变噪声水平。它让样本从一个更模糊的状态（高 $t$）演化到一个稍清晰的状态（低 $t$）。
- **问题**：由于这一步跨度较大，且分数模型 $\mathbf{s}_\theta$ 存在预测偏差，$\mathbf{x}_{t-\Delta t}$ 往往会产生数值漂移，导致样本点偏离真实的数据流形。

---

### 2. Corrector (校正器)：定点提纯

Corrector 的作用是在时间步固定的前提下，对样本进行微调。它不改变 $t$，只改变 $\mathbf{x}$ 的保真度。

#### 核心算法：退火朗之万动力学 (Annealed Langevin Dynamics)

迭代方程：在 $t-\Delta t$ 时刻，进行 $M$ 步如下操作：
$$
\mathbf{x}_{i+1} = \mathbf{x}_i + \epsilon \mathbf{s}_\theta(\mathbf{x}_i, t-\Delta t) + \sqrt{2\epsilon} \mathbf{z}
$$

- $\epsilon$ 是很小的步长。

#### 内在逻辑：
- **梯度爬坡**：$\mathbf{s}_\theta$ 指向当前噪声水平下概率密度最高的方向。Corrector 顺着这个方向推样本。
- **噪声平衡**：注入 $\sqrt{2\epsilon} \mathbf{z}$ 是为了模拟真实的概率分布，防止样本全部坍缩到局部最大值，使其符合当前 $t$ 时刻的分布特征。

- **目的**：“纠偏”。它相当于把 Predictor 甩出去的样本点重新“拉回”到当前概率分布的中心轨道上。

---

### 3. PC 采样器的完整迭代算法

在实际操作中，每一时间步都是 Predictor 和 Corrector 的嵌套循环：

#### 算法流程：
输入：训练好的分数模型 $\mathbf{s}_\theta$，初始噪声 $\mathbf{x}_T \sim \mathcal{N}(0, \mathbf{I})$

```python
FOR t = T down to 1 DO:
    # Predictor 步
    根据离散反向 SDE 算出 \mathbf{x}_{t-1}。
    此时时间从 t 变到了 t-1。

    # Corrector 步
    FOR m = 1 to M DO:
        利用朗之万动力学在 t-1 的刻度下微调 \mathbf{x}_{t-1}。
    END FOR
END FOR
```

输出：生成的数据样本 $\mathbf{x}_0$

---

### 4. 为什么要用 PC 采样器？

- **容错性**：如果只用 Predictor（如单纯的 DDPM 步进），一旦某一步走偏了，误差会不断累积，最后生成模糊或有伪影的图片。Corrector 在每一步都及时止损。
- **样本质量**：Corrector 增加了采样的计算量（每一步多算 $M$ 次网络前向），但换来了更高的样本保真度（Fidelity）。它保证了在每一个噪声级别上，样本都达到了平衡态。
- **灵活性**：Predictor 负责“宏观结构”，Corrector 负责“微观纯化”。你可以根据需求调整 $M$ 的大小——需要高质量时调大 $M$，需要速度时调小 $M$。

---

### 总结

PC 采样器就是一个**“步进 + 修正”**的过程。

- **Predictor** 像是在浓雾中向目标方向迈出一大步，
- **Corrector** 则是停下来，在原地仔细辨别方位并细微调整站姿，确保自己没有走偏，然后再准备迈出下一大步。

## DPM-Solver：高效采样算法

DPM-Solver 的出现是扩散模型采样技术的一个里程碑。它的核心逻辑是：既然反向过程是一个 ODE/SDE，我们能不能利用解析解（Exact Solution）和高阶数值方法，把采样步数压缩到 10-20 步，且不损失质量？

---

### 1. 核心洞察：将 ODE 拆解为“线性”与“非线性”

DPM-Solver 的推导始于对 **概率流 ODE (Probability Flow ODE)** 的重新观察。对于 VP SDE，其对应的 ODE 可以写成如下形式：
$$
d\mathbf{x} = \underbrace{f(t)\mathbf{x}}_{\text{线性项}} dt + \underbrace{h(t)\mathbf{s}_\theta(\mathbf{x}, t)}_{\text{非线性项}} dt
$$

- 如果直接用欧拉法（Euler Method），误差会很大。
- DPM-Solver 做了一个非常聪明的变换：
  1. **解析处理线性项**：线性项（即漂移项）描述的是均值的缩放，这部分是可以求出解析解的（就像我们在 VP 推导中看到的指数衰减）。
  2. **积分变换**：通过变量代换（将时间 $t$ 变换为对数信噪比 $\lambda$），可以将复杂的 ODE 转化为一个相对平滑的积分形式。

---

### 2. DPM-Solver 的解法思路

#### A. 变化率的特殊性
在扩散模型中，数据变化最剧烈的地方是在信噪比较高（$t$ 较小）的时候。DPM-Solver 并不是在时间轴 $t$ 上均匀取点，而是在对数信噪比 $\lambda$ 空间内进行操作。

#### B. 高阶泰勒展开 (Order-k)
- **DPM-Solver-1 (一阶)**：本质上是解析版的 DDIM。它假设在一个极小的步长内，模型预测的噪声（或分数）是恒定的。
- **DPM-Solver-2 (二阶)**：这是最常用的版本。它假设模型预测的噪声随时间线性变化。为了计算这个变化率，它会在两个时间点采样，然后利用类似龙格-库塔（Runge-Kutta）的方法进行拟合。
- **DPM-Solver-3 (三阶)**：进一步假设噪声是二阶变化的。

---

### 3. 为什么 DPM-Solver 这么快？

- **解析解保证基准**：因为它对线性部分采用了解析解，所以在步长较大时，它依然能保证样本处于大致正确的尺度上，不会像 PC 采样器那样轻易偏离流形。
- **高阶精度**：普通采样器（如 DDPM）是一阶精度的，误差随步长 $\Delta t$ 线性增长；而 DPM-Solver-2 是二阶精度的，误差随 $(\Delta t)^2$ 增长。这意味着你可以用更少的步数达到相同的精度。
- **无须额外训练**：它是一个纯粹的数学加速算子，直接作用于你训练好的模型。

---

### 4. DPM-Solver vs DPM-Solver++

你在现在的 Stable Diffusion 界面里经常看到 DPM++ 2M 或 DPM++ SDE，这是它们的进阶版：

- **DPM-Solver++**：针对数据预测（x-prediction）进行了优化。在极低步数（如 5-10 步）下，生成的图像比原版更清晰，不容易出现伪影。
- **2M (Multistep)**：利用前面已经算过的步数信息来预测下一步，不需要在每一步都多次运行模型，进一步节省了推理时间。
- **SDE 版本**：在 ODE 路径中重新引入了一点随机性。这能增加画面的细节，解决 ODE 采样有时会导致的“过于平滑”或“颜色灰暗”的问题。

---

### 总结：从 PC 到 DPM-Solver

- **PC 采样器**：像是一个谨慎的工匠。走一小步，停下来磨一磨，再走一小步。稳健但极慢。
- **DPM-Solver**：像是一个数学家。他算出了路径的解析规律，即使闭着眼跨出一大步，也能通过二阶修正精确踩在预定的点上。

## VP 与 VE 的基础方程及概率流 ODE 推导

### 1. VP 与 VE 的基础方程

#### **VP (Variance Preserving) 策略**
常用于 DDIM / Stable Diffusion。其特点是数据的方差在加噪过程中始终保持为 1。
- **前向加噪 SDE:**
$$
d\mathbf{x} = -\frac{1}{2}\beta(t)\mathbf{x} dt + \sqrt{\beta(t)} d\mathbf{w}
$$
- **反向去噪 SDE:**
$$
d\mathbf{x} = \left[-\frac{1}{2}\beta(t)\mathbf{x} - \beta(t) \nabla_{\mathbf{x}} \log p_t(\mathbf{x})\right] dt + \sqrt{\beta(t)} d\bar{\mathbf{w}}
$$

#### **VE (Variance Exploding) 策略**
常用于 Score-based models (SMLD)。其特点是方差随时间无限增长。
- **前向加噪 SDE:**
$$
d\mathbf{x} = \sqrt{\frac{d[\sigma^2(t)]}{dt}} d\mathbf{w}
$$
- **反向去噪 SDE:**
$$
d\mathbf{x} = \left[-\frac{d[\sigma^2(t)]}{dt} \nabla_{\mathbf{x}} \log p_t(\mathbf{x})\right] dt + \sqrt{\frac{d[\sigma^2(t)]}{dt}} d\bar{\mathbf{w}}
$$

---

### 2. 从反向 SDE 推导概率流 ODE

#### 核心理论
任何扩散 SDE：
$$
d\mathbf{x} = \mathbf{f}(\mathbf{x}, t)dt + g(t)d\mathbf{w}
$$
都存在一个对应的**概率流 ODE**，其轨迹的概率密度演化与 SDE 完全一致：
$$
\frac{d\mathbf{x}}{dt} = \mathbf{f}(\mathbf{x}, t) - \frac{1}{2}g(t)^2 \nabla_{\mathbf{x}} \log p_t(\mathbf{x})
$$

#### 推导步骤

1. **代入得分函数 (Score Function)**
   利用神经网络训练出的噪声预测模型 $\epsilon_\theta(\mathbf{x}_t, t)$ 与得分函数的关系：
   $$
   \nabla_{\mathbf{x}} \log p_t(\mathbf{x}) = -\frac{\epsilon_\theta(\mathbf{x}_t, t)}{\sigma_t}
   $$
   其中 $\sigma_t$ 是 $t$ 时刻噪声的标准差。

2. **针对 VP 模型推导**
   对于 VP 模型，我们已知 $\mathbf{f}(\mathbf{x}, t) = -\frac{1}{2}\beta(t)\mathbf{x}$ 且 $g(t)^2 = \beta(t)$。
   代入 ODE 通式：
   $$
   \frac{d\mathbf{x}_t}{dt} = -\frac{1}{2}\beta(t)\mathbf{x}_t - \frac{1}{2}\beta(t) \left[ -\frac{\epsilon_\theta(\mathbf{x}_t, t)}{\sigma_t} \right]
   $$
   整理得：
   $$
   \frac{d\mathbf{x}_t}{dt} = \underbrace{-\frac{1}{2}\beta(t)}_{f(t)} \mathbf{x}_t + \underbrace{\frac{\beta(t)}{2\sigma_t}}_{g(t)} \epsilon_\theta(\mathbf{x}_t, t)
   $$

3. **系数的物理意义**
   为了让公式更具通用性（兼容 VP 和 VE），我们可以利用 $\alpha_t$ 和 $\sigma_t$ 的导数来表达：
   - **线性项系数:** $f(t) = \frac{d \log \alpha_t}{dt}$。在 VP 中这等于 $-\frac{1}{2}\beta(t)$。
   - **噪声项系数:** $g(t) = \sigma_t \frac{d \lambda_t}{dt}$（其中 $\lambda_t = \log(\alpha_t/\sigma_t)$ 是 log-SNR）。

---

### 3. 总结推导后的统一形式

无论 VP 还是 VE，最终都可以写成 DPM-Solver 所使用的**半线性形式**：
$$
dx_t = f(t)x_t dt + g(t)\epsilon_\theta(x_t, t) dt
$$

- **VP 模式下:** $f(t) = \frac{d \log \alpha_t}{dt}$, $g(t) = -\alpha_t \frac{d(\sigma_t/\alpha_t)}{dt}$。
- **VE 模式下:** $f(t) = 0$（因为 VE 的漂移项为 0）, $g(t) = \frac{1}{2\sigma_t} \frac{d\sigma_t^2}{dt}$。

**推导的核心意义：** 通过这种变换，我们将随机的去噪过程变成了确定性的斜率积分问题。公式左边的 $f(t)x_t$ 是可以精确求解的指数部分，而右边的 $\epsilon_\theta$ 项则是我们要用高阶数值方法（如泰勒展开）去近似的非线性部分。这正是 DPM-Solver 能够单步大跨度采样的数学前提。

## 通用 ODE 的解析解：积分因子法

对于通用的 ODE 形式：
$$
\frac{d\mathbf{x}_t}{dt} = f(t)\mathbf{x}_t + g(t)\epsilon_\theta(\mathbf{x}_t, t)
$$

我们可以通过引入积分因子 (Integrating Factor) 来求解。

---

### 第一步：引入积分因子 (Integrating Factor)

对于这种形式的线性非齐次 ODE，解的标准做法是引入积分因子 $\Phi(t)$。定义：
$$
\Phi(t) = \exp\left( -\int f(t) dt \right)
$$
它满足：
$$
\frac{d\Phi(t)}{dt} = -f(t)\Phi(t)
$$

将 ODE 两边同时乘以 $\Phi(t)$：
$$
\Phi(t) \frac{d\mathbf{x}_t}{dt} - \Phi(t) f(t) \mathbf{x}_t = \Phi(t) g(t) \epsilon_\theta(\mathbf{x}_t, t)
$$

注意到左边正好是乘积求导的逆过程：
$$
\frac{d}{dt} \left[ \Phi(t) \mathbf{x}_t \right] = \Phi(t) g(t) \epsilon_\theta(\mathbf{x}_t, t)
$$

---

### 第二步：积分并解析表达

从 $s$（起始时刻）积分到 $t$（终止时刻）：
$$
\Phi(t) \mathbf{x}_t - \Phi(s) \mathbf{x}_s = \int_s^t \Phi(\tau) g(\tau) \epsilon_\theta(\mathbf{x}_\tau, \tau) d\tau
$$

解出 $\mathbf{x}_t$：
$$
\mathbf{x}_t = \frac{\Phi(s)}{\Phi(t)} \mathbf{x}_s + \frac{1}{\Phi(t)} \int_s^t \Phi(\tau) g(\tau) \epsilon_\theta(\mathbf{x}_\tau, \tau) d\tau
$$

---

## 引入 $\lambda$ 的动机

要理解引入 $\lambda$ 的动机，我们必须先看清在这个公式下，如果我们**不引入**新变量，会遇到什么样无法逾越的工程和数学障碍。

---

### 1. 现实困境：积分项中的“黑盒”耦合

在这个积分中，$\epsilon_\theta$ 是一个神经网络，它是关于 $\tau$ 的复合函数 $\epsilon_\theta(\mathbf{x}_\tau, \tau)$。
我们的目标是对这个积分进行数值离散化（比如用矩形法或梯形法）。但问题在于积分核：
$$K(\tau) = \Phi(\tau) g(\tau)$$
这个 $K(\tau)$ 取决于你选择的噪声策略（VP, VE, 或各种线性/余弦调度）。

- **问题 A（系数剧烈波动）：** 在时间轴 $t$ 上，$\Phi(\tau)$（积分因子）和 $g(\tau)$ 通常是指数级变化的。例如在 VP 模型中，它们包含复杂的 $\exp(\int \beta(t)dt)$。这意味着在积分区间 $[s, t]$ 内，权重分布极度不均匀，直接对 $\epsilon_\theta$ 做线性近似会导致巨大的截断误差。
- **问题 B（步长无法统一）：** 不同的 $t$ 步长下，$\epsilon_\theta$ 的演化速率完全不同。你很难找到一个通用的 $dt$ 能同时适配去噪初期（低 SNR）和后期（高 SNR）。

---

### 2. 核心动机：寻找“线性化”的轴

引入 $\lambda$ 的核心动机是**解耦**。我们希望找到一个变换 $\tau \to \lambda$，使得积分项中的系数部分能被“吃掉”，或者变成一个**与具体噪声调度无关**的标准形式。

我们观察积分项中的微分部分：$K(\tau) d\tau = \Phi(\tau) g(\tau) d\tau$。
**如果能定义一个 $\lambda$，使得：**
$$\frac{d\lambda}{d\tau} = \text{某种能抵消 } K(\tau) \text{ 的形式}$$

那么，原本复杂的变系数积分就会变成一个**常系数**或**标准指数形式**的积分。

---

### 3. 为什么必须是 $\lambda = \ln(\alpha_t / \sigma_t)$？

这不仅是一个定义，这是一个**数学上的必然选择**。

#### A. 实现“半线性解”的完美对齐

在扩散模型中，$\alpha_t$ 代表信号强度，$\sigma_t$ 代表噪声强度。
如果我们定义 $\lambda = \ln(\alpha_t / \sigma_t)$，你会发现一个神奇的性质：
在所有符合概率流 ODE 定义的模型中，积分因子 $\Phi(\tau)$、扩散系数 $g(\tau)$ 以及 $d\tau$ 的乘积，经过变量替换后，其系数项会化简为：
$$\Phi(\tau) g(\tau) d\tau \propto e^{-\lambda} d\lambda$$

这导致了一个极其重要的结果：**所有的噪声调度（无论多复杂），在 $\lambda$ 轴上看起来都一模一样。** 它们都变成了一个简单的指数衰减权重。

#### B. 神经网络在 $\lambda$ 空间最平滑

这是数值计算上的动机。如果你在 $t$ 轴上画出神经网络的输出 $\epsilon_\theta$，它往往在某些阶段变化剧烈，在某些阶段非常平坦。
但如果你在 $\lambda$（log-SNR）轴上观察 $\epsilon_\theta$，你会发现它几乎是**近乎线性**变化的。

> **结论：** 在一个近乎线性的空间做泰勒展开（DPM-Solver 的核心），比在非线性空间做展开，精度要高出几个数量级。

---

### 4. 总结：如果不加 $\lambda$ 会怎样？

如果我们坚持在原公式的 $t$ 轴上操作：
1.  **无法写出通用的 Solver：** 你必须为每一种 $\beta(t)$ 调度手动计算一套复杂的积分权重。
2.  **步数效率极低：** 为了捕捉 $K(\tau)$ 的剧烈变化，你被迫使用极小的步长。
3.  **失去指数积分器的优势：** 你无法利用 $e^h$ 这种解析解来抵消掉漂移项（Drift term）带来的误差。

**引入 $\lambda$ 的本质动机是将“病态的、随调度变化的变系数 ODE”转化为“良性的、统一的常系数 ODE”。** 这一步跨越，让采样步数从 50 步降低到 10 步成为了可能。

## 从扩散模型的基础定义推导 ODE

### 1. 扩散模型的基础定义

在任何扩散模型（如 DDPM 或连续 SDE）中，我们定义在时刻 $t$ 的样本 $\mathbf{x}_t$ 是由原始数据 $\mathbf{x}_0$ 和标准高斯噪声 $\mathbf{\epsilon}$ 组合而成的：
$$
\mathbf{x}_t = \alpha_t \mathbf{x}_0 + \sigma_t \mathbf{\epsilon}
$$
其中：
- $\alpha_t$ 是信号保留系数（决定了还剩下多少原始信息）。
- $\sigma_t$ 是噪声系数（决定了注入了多少随机性）。

---

### 2. 对基础定义求导

为了得到 ODE 的形式 $\frac{d\mathbf{x}_t}{dt}$，我们对上面的定义式关于时间 $t$ 求全导数：
$$
\frac{d\mathbf{x}_t}{dt} = \frac{d\alpha_t}{dt} \mathbf{x}_0 + \frac{d\sigma_t}{dt} \mathbf{\epsilon}
$$

---

### 3. 利用消元法构建 ODE

现在的目标是把右边的 $\mathbf{x}_0$ 和 $\mathbf{\epsilon}$ 消掉，换成当前时刻的变量 $\mathbf{x}_t$。从基础定义中，我们可以解出 $\mathbf{x}_0$：
$$
\mathbf{x}_0 = \frac{\mathbf{x}_t - \sigma_t \mathbf{\epsilon}}{\alpha_t}
$$

将 $\mathbf{x}_0$ 代入刚才的导数式：
$$
\frac{d\mathbf{x}_t}{dt} = \frac{d\alpha_t}{dt} \left( \frac{\mathbf{x}_t - \sigma_t \mathbf{\epsilon}}{\alpha_t} \right) + \frac{d\sigma_t}{dt} \mathbf{\epsilon}
$$

整理各项，把含有 $\mathbf{x}_t$ 的项放在一起：
$$
\frac{d\mathbf{x}_t}{dt} = \left( \frac{1}{\alpha_t} \frac{d\alpha_t}{dt} \right) \mathbf{x}_t + \left( \frac{d\sigma_t}{dt} - \frac{\sigma_t}{\alpha_t} \frac{d\alpha_t}{dt} \right) \mathbf{\epsilon}
$$

---

### 4. 识别漂移项 $f(t)$

对比标准 ODE 形式 $\frac{d\mathbf{x}_t}{dt} = f(t)\mathbf{x}_t + (\dots)$，我们可以清楚地看到 $\mathbf{x}_t$ 前面的系数就是：
$$
f(t) = \frac{1}{\alpha_t} \frac{d\alpha_t}{dt}
$$

根据微积分的基本公式 $\frac{d \ln u}{dx} = \frac{1}{u} \frac{du}{dx}$，上式完全等价于：
$$
f(t) = \frac{d \ln \alpha_t}{dt}
$$

### 总结
1. **动力学模型 (SDE)**

前向加噪过程定义为线性漂移 SDE：
$$
dx = f(t)x dt + g(t) d\mathbf{w}
$$

2. **参数解析映射**

利用积分因子 $\alpha_t = \exp \left( \int_0^t f(s) ds \right)$ 求解，得到 $x(t)$ 的闭式解：

| 物理量符号 | 函数表达式 (与系数 $f$, $g$ 的关系) |
|------------|------------------------------------|
| **信号缩放** $\alpha_t$ | $\exp \left( \int_0^t f(s) ds \right)$ |
| **噪声强度** $\sigma_t^2$ | $\alpha_t^2 \int_0^t \frac{g(s)^2}{\alpha_s^2} ds$ |
| **漂移项系数** $f(t)$ | $\frac{d \log \alpha_t}{dt}$ |
| **扩散项系数** $g^2(t)$ | $\frac{d\sigma_t^2}{dt} - 2 f(t) \sigma_t^2$ |

3. **一步加噪公式 (Reparameterization)**

得益于线性高斯性质，可以直接从 $x(0)$ 计算任意时刻 $x(t)$：
$$
x(t) = \alpha_t x(0) + \sigma_t \epsilon, \quad \epsilon \sim \mathcal{N}(0, \mathbf{I})
$$

### 5. 引入积分因子 $\Phi(t)$ 并简化公式

我们之前定义的积分因子：
$$
\Phi(t) = \exp\left(-\int f(t)dt\right)
$$

代入 $f(t)$ 的表达式：
$$
\Phi(t) = \exp\left(-\int \frac{d \ln \alpha_t}{dt} dt \right) = \exp(-\ln \alpha_t) = \frac{1}{\alpha_t}
$$

将其代入通用积分公式：
$$
\mathbf{x}_t = \frac{\Phi(s)}{\Phi(t)} \mathbf{x}_s + \frac{1}{\Phi(t)} \int_s^t \Phi(\tau) g(\tau) \epsilon_\theta(\mathbf{x}_\tau, \tau) d\tau
$$

由于 $\Phi(t) = 1/\alpha_t$，公式可以进一步简化：
$$
\mathbf{x}_t = \frac{1/\alpha_s}{1/\alpha_t} \mathbf{x}_s + \alpha_t \int_s^t \frac{1}{\alpha_\tau} g(\tau) \epsilon_\theta(\mathbf{x}_\tau, \tau) d\tau
$$

最终得到：
$$
\mathbf{x}_t = \frac{\alpha_t}{\alpha_s} \mathbf{x}_s + \alpha_t \int_s^t \frac{g(\tau)}{\alpha_\tau} \epsilon_\theta(\mathbf{x}_\tau, \tau) d\tau
$$

## 从 Fokker-Planck 方程（FPE）推导二阶矩演化公式

### 1. 写出 FPE

对于前向加噪 SDE：
$$d\mathbf{x}_t = f(t)\mathbf{x}_t dt + g(t)d\mathbf{w}$$

其概率密度的演化方程为：
$$
\frac{\partial p}{\partial t} = -\frac{\partial}{\partial x} [f(t)x p] + \frac{1}{2} g(t)^2 \frac{\partial^2 p}{\partial x^2}
$$

---

### 2. 构造二阶矩的定义

二阶矩 $\Sigma_t$ 定义为：
$$
\Sigma_t = \int x^2 p(x, t) dx
$$

我们对时间 $t$ 求导，并将 FPE 代入：
$$
\frac{d\Sigma_t}{dt} = \int x^2 \frac{\partial p}{\partial t} dx = \int x^2 \left( -\frac{\partial}{\partial x} [f x p] + \frac{1}{2} g^2 \frac{\partial^2 p}{\partial x^2} \right) dx
$$

---

### 3. 分步积分（Integration by Parts）

#### 第一项（漂移项贡献）

使用分部积分 $\int u dv = uv - \int v du$，令 $u = x^2, dv = \frac{\partial}{\partial x}(fxp) dx$：
$$
\int x^2 \left( -\frac{\partial}{\partial x} [f x p] \right) dx = \underbrace{\left[ -x^2 f x p \right]_{-\infty}^{\infty}}_{0} + \int 2x (f x p) dx = 2f(t) \int x^2 p dx = 2f(t)\Sigma_t
$$

#### 第二项（扩散项贡献）

同样使用分部积分（需两次），处理 $\frac{\partial^2 p}{\partial x^2}$：
$$
\frac{1}{2} g^2 \int x^2 \frac{\partial^2 p}{\partial x^2} dx = \underbrace{\left[ \text{边界项} \right]}_{0} - \frac{1}{2} g^2 \int 2x \frac{\partial p}{\partial x} dx = \underbrace{\left[ \text{边界项} \right]}_{0} + \frac{1}{2} g^2 \int 2 p dx
$$

由于概率密度积分 $\int p dx = 1$，此项简化为：
$$
g(t)^2
$$

---

### 4. 合并结果

将两部分结果合并：
$$
\frac{d\Sigma_t}{dt} = 2f(t)\Sigma_t + g(t)^2
$$

## 核心约束：方差演化方程

### 1. 方差演化方程

考虑一个通用的前向加噪 SDE：
$$
d\mathbf{x}_t = f(t)\mathbf{x}_t dt + g(t)d\mathbf{w}
$$
其中 $f(t) = \frac{d \ln \alpha_t}{dt}$。我们要研究这个系统随时间演化的方差 $\Sigma_t$。对于这种线性 SDE，方差的演化遵循一个确定的微分方程（来自矩演化方程或 Fokker-Planck 方程的二阶矩推导）：
$$
\frac{d\Sigma_t}{dt} = 2f(t)\Sigma_t + g(t)^2
$$

---

### 2. 物理对齐：代入预设的方差

在扩散模型中，我们预先定义了时刻 $t$ 的边际分布是 $\mathcal{N}(\alpha_t \mathbf{x}_0, \sigma_t^2 \mathbf{I})$。这意味着，我们“钦定”了该 SDE 的方差必须是：
$$
\Sigma_t = \sigma_t^2
$$

为了让这个预设成立，我们将 $\Sigma_t = \sigma_t^2$ 代入步骤 1 的演化方程中：
$$
\frac{d(\sigma_t^2)}{dt} = 2f(t)\sigma_t^2 + g(t)^2
$$

---

### 3. 解出 $g(t)$ 的特定关系

现在我们直接把 $g(t)^2$ 孤立出来：
$$
g(t)^2 = \frac{d\sigma_t^2}{dt} - 2f(t)\sigma_t^2
$$

这就是 $g(t)$ 必须满足的基本微分关系。

## 利用“概率流守恒”约束简化积分

### 1. $g(t)$ 的微分关系

在概率流 ODE 中，$g(t)$ 不是独立定义的，它必须满足与 $\sigma_t$ 的特定微分关系：
$$
g(t)^2 = \frac{d\sigma_t^2}{dt} - 2f(t)\sigma_t^2
$$

我们将此式变形，提取出 $g(t)$：
$$
g(t) = \sqrt{\frac{d\sigma_t^2}{dt} - 2\frac{d \ln \alpha_t}{dt}\sigma_t^2} = \sigma_t \sqrt{\frac{d \ln \sigma_t^2}{dt} - 2\frac{d \ln \alpha_t}{dt}}
$$

进一步化简：
$$
g(t) = \sigma_t \sqrt{2 \left( \frac{d \ln \sigma_t}{dt} - \frac{d \ln \alpha_t}{dt} \right)} = \sigma_t \sqrt{-2 \frac{d}{dt} \ln \left( \frac{\alpha_t}{\sigma_t} \right)}
$$

由于 $\lambda = \ln(\alpha_t / \sigma_t)$，我们得到：
$$
g(t) = \sigma_t \sqrt{-2 \frac{d\lambda}{dt}}
$$

---

### 2. 变量替换 $dt \to d\lambda$

现在我们将积分项 $I = \int_s^t \frac{g(\tau)}{\alpha_\tau} \epsilon_\theta d\tau$ 中的 $dt$ 换掉。由 $\lambda = \ln \alpha_t - \ln \sigma_t$ 得：
$$
d\lambda = \left( \frac{d \ln \alpha_t}{dt} - \frac{d \ln \sigma_t}{dt} \right) dt
$$

注意到在这个微分关系下，结合步骤 1 的结论，我们可以推导出：
$$
\frac{g(\tau)}{\alpha_\tau} d\tau = \frac{\sigma_\tau \sqrt{-2 \frac{d\lambda}{d\tau}}}{\alpha_\tau} d\tau
$$

#### 最神奇的一步：
在 $\lambda$ 域内，为了让 ODE 变成半线性常系数形式，DPM-Solver 证明了在所有的扩散 ODE 中，这个组合项 $g(\tau) d\tau$ 与信号项的比例恰好满足：
$$
\frac{g(\tau)}{\alpha_\tau} d\tau = -\frac{\sigma_\tau}{\alpha_\tau} d\lambda
$$

(注：这里利用了 $d\lambda$ 与 $d\tau$ 的平方根关系以及系数抵消，详细的链式法则展开会发现 $\sqrt{-2\dot{\lambda}}$ 项在积分元变换中被平滑掉了)

由于 $e^{-\lambda} = \frac{\sigma_\tau}{\alpha_\tau}$，代入上式：
$$
\frac{g(\tau)}{\alpha_\tau} d\tau = -e^{-\lambda} d\lambda
$$

---

### 3. 最终结果：系数全部“消失”

将 $-e^{-\lambda} d\lambda$ 代回积分式 $I$：
$$
I = \alpha_t \int_{\lambda_s}^{\lambda_t} (-e^{-\lambda}) \epsilon_\theta(\lambda) d\lambda
$$

交换积分上下限（消除负号）：
$$
I = -\alpha_t \int_{\lambda_s}^{\lambda_t} e^{-\lambda} \epsilon_\theta(\lambda) d\lambda
$$

利用 $\alpha_t = \sigma_t e^{\lambda_t}$，将 $\alpha_t$ 移入积分号：
$$
I = - \sigma_t e^{\lambda_t} \int_{\lambda_s}^{\lambda_t} e^{-\lambda} \epsilon_\theta(\lambda) d\lambda = - \sigma_t \int_{\lambda_s}^{\lambda_t} e^{\lambda_t - \lambda} \epsilon_\theta(\lambda) d\lambda
$$

---

### 4. 总结

复杂的 $g(\tau), \Phi(\tau)$ 以及非线性的 $dt$ 是如何消掉的？
- $\Phi(\tau)$ 变成了 $1/\alpha_\tau$。
- $g(\tau)$ 被分解为包含 $\sqrt{\dot{\lambda}}$ 的形式。
- 变量替换 $dt \to d\lambda$ 产生的雅可比行列式（Jacobian）正好与 $g(\tau)$ 中的复杂项抵消。

最后剩下的只有 $\sigma_\tau / \alpha_\tau$，而它在定义上就是 $e^{-\lambda}$。这样，原本依赖于具体噪声策略（$\beta(t)$）的复杂积分，就变成了只依赖于相对信噪比距离（$\lambda_t - \lambda$）的指数积分。这就是 DPM-Solver 实现“跨调度统一”的数学底牌。

## DPM-Solver 的数值积分策略

### 1. 泰勒展开近似

要计算积分 $I = \int_{\lambda_s}^{\lambda_t} e^{\lambda_t - \lambda} \epsilon_\theta(\lambda) d\lambda$，我们面临的核心挑战是：神经网络 $\epsilon_\theta(\lambda)$ 是一个“黑盒”，没有解析表达式。

为了求出数值解，DPM-Solver 采用了数值积分中经典的**多项式逼近策略**。其核心思想是：在很小的积分区间 $[\lambda_s, \lambda_t]$ 内，假设 $\epsilon_\theta(\lambda)$ 的变化是平滑的，可以用**泰勒展开**来近似它。

我们在起始点 $\lambda_s$ 处对 $\epsilon_\theta(\lambda)$ 进行 $k$ 阶展开：
$$
\epsilon_\theta(\lambda) \approx \epsilon_\theta(\lambda_s) + \epsilon_\theta'(\lambda_s)(\lambda - \lambda_s) + \frac{\epsilon_\theta''(\lambda_s)}{2}(\lambda - \lambda_s)^2 + \dots
$$

将展开式代入积分 $I$ 中：
$$
I = -\sigma_t \int_{\lambda_s}^{\lambda_t} e^{\lambda_t - \lambda} \left[ \sum_{n=0}^{k-1} \frac{\epsilon_\theta^{(n)}(\lambda_s)}{n!}(\lambda - \lambda_s)^n \right] d\lambda
$$

由于求和与积分可以交换，计算 $I$ 变成了计算一系列形式为 $\int e^{\lambda_t - \lambda} (\lambda - \lambda_s)^n d\lambda$ 的基础积分。

---

### 2. 核心积分项的解析解 (The $\phi_n$ functions)

定义步长 $h = \lambda_t - \lambda_s$。通过分部积分法，我们可以解析地算出这些项：

- **$n=0$ (一阶项):**
    $$
    \int_{\lambda_s}^{\lambda_t} e^{\lambda_t - \lambda} d\lambda = \left[ -e^{\lambda_t - \lambda} \right]_{\lambda_s}^{\lambda_t} = e^{\lambda_t - \lambda_s} - 1 = e^h - 1
    $$
- **$n=1$ (二阶项):**
    $$
    \int_{\lambda_s}^{\lambda_t} e^{\lambda_t - \lambda} (\lambda - \lambda_s) d\lambda = e^h - h - 1
    $$

在 DPM-Solver 的论文和代码实现中，这些系数被统一定义为 $\phi_n(h)$ 函数。例如：
- $\phi_1(h) = e^h - 1$
- $\phi_2(h) = \frac{e^h - h - 1}{h}$

---

### 3. 不同阶数的求解方案

#### **DPM-Solver-1 (一阶方案)**

只保留泰勒展开的第一项（常数项），假设 $\epsilon_\theta(\lambda) \approx \epsilon_\theta(\lambda_s)$：
$$
I \approx -\sigma_t \epsilon_\theta(\lambda_s) \int_{\lambda_s}^{\lambda_t} e^{\lambda_t - \lambda} d\lambda = -\sigma_t (e^h - 1) \epsilon_\theta(\lambda_s)
$$

这就是**一阶更新公式**。它在数学形式上与 DDIM 非常接近，但由于是在 $\lambda$ 域积分，精度更高。

#### **DPM-Solver-2 (二阶方案)**

保留到一阶导数项。由于我们无法直接得到神经网络的导数 $\epsilon_\theta'(\lambda_s)$，DPM-Solver 采用**中点采样**来近似导数：
1.  先用一阶公式走半步，预测一个中间点 $\mathbf{x}_{s1}$。
2.  在中间点调用模型得到 $\epsilon_\theta(\lambda_{s1})$。
3.  利用有限差分估算斜率：$\epsilon_\theta' \approx \frac{\epsilon_\theta(\lambda_{s1}) - \epsilon_\theta(\lambda_s)}{\lambda_{s1} - \lambda_s}$。

将这个斜率代回积分式，就能得到二阶更新公式，它比一阶方案能承载更多的轨迹曲率信息。

---

### 4. 总结计算逻辑

计算这个积分的步骤在代码中体现为：
1.  **确定步长 $h$：** 根据当前的 $t$ 和目标 $t'$ 计算 $\lambda$ 的差值。
2.  **获取模型输出：** 得到当前的噪声预测 $\epsilon_\theta$。
3.  **计算系数：** 算出 $e^h - 1$ 等系数。
4.  **线性组合：** 将前一时刻的 $x_s$、当前的 $\epsilon_\theta$ 按照推导出的系数加权求和，得到 $x_t$。

#### **为什么这样算很快？**
因为 $e^h - 1$ 这种系数只取决于时间步，可以预先算好。每一步采样只需要调用 1-2 次神经网络，却能达到普通 Euler 法几十次采样才能达到的精度。

## DPM-Solver++ 的改进与多步法（符号修正 + 详细推导）

DPM-Solver++ 的核心是把模型输出统一为 $\hat{\mathbf{x}}_0$，并在 $\lambda$ 域做高阶积分。

先统一记号：
- $\lambda = \log(\alpha/\sigma)$。
- 采样步长 $h = \lambda_t - \lambda_s$（通常去噪时 $h>0$）。
- 记 $\hat{\mathbf{x}}_\theta(\lambda)$ 为网络在时刻 $\lambda$ 的 $x_0$ 预测。

---

### 1. 从 $\epsilon$ 形式到 $x_0$ 形式：完整替换过程

我们从 DPM-Solver 在 $\lambda$ 域的噪声积分式出发：
$$
\mathbf{x}_t = \frac{\alpha_t}{\alpha_s}\mathbf{x}_s - \alpha_t \int_{\lambda_s}^{\lambda_t} e^{-\lambda}\,\epsilon_\theta(\lambda)\,d\lambda
$$

> 等价写法是
$$
\mathbf{x}_t = \frac{\alpha_t}{\alpha_s}\mathbf{x}_s - \sigma_t \int_{\lambda_s}^{\lambda_t} e^{\lambda_t-\lambda}\,\epsilon_\theta(\lambda)\,d\lambda
$$
因为 $\alpha_t e^{-\lambda}=\sigma_t e^{\lambda_t-\lambda}$。

#### 第 1 步：$\epsilon_\theta$ 与 $\hat{\mathbf{x}}_0$ 的关系

由
$$
\mathbf{x}_\lambda = \alpha_\lambda \hat{\mathbf{x}}_\theta(\lambda)+\sigma_\lambda\epsilon_\theta(\lambda)
$$
可得
$$
\epsilon_\theta(\lambda)=\frac{\mathbf{x}_\lambda-\alpha_\lambda\hat{\mathbf{x}}_\theta(\lambda)}{\sigma_\lambda}
=\frac{\mathbf{x}_\lambda}{\sigma_\lambda}-\frac{\alpha_\lambda}{\sigma_\lambda}\hat{\mathbf{x}}_\theta(\lambda)
=\frac{\mathbf{x}_\lambda}{\sigma_\lambda}-e^{\lambda}\hat{\mathbf{x}}_\theta(\lambda)
$$

#### 第 2 步：代回积分并化简被积函数

利用 $e^{-\lambda}=\sigma_\lambda/\alpha_\lambda$，有
$$
e^{-\lambda}\epsilon_\theta(\lambda)
=\frac{\sigma_\lambda}{\alpha_\lambda}\left(\frac{\mathbf{x}_\lambda}{\sigma_\lambda}-e^{\lambda}\hat{\mathbf{x}}_\theta(\lambda)\right)
=\frac{\mathbf{x}_\lambda}{\alpha_\lambda}-\hat{\mathbf{x}}_\theta(\lambda)
$$

所以
$$
\mathbf{x}_t
=\frac{\alpha_t}{\alpha_s}\mathbf{x}_s
-\alpha_t\int_{\lambda_s}^{\lambda_t}\left(\frac{\mathbf{x}_\lambda}{\alpha_\lambda}-\hat{\mathbf{x}}_\theta(\lambda)\right)d\lambda
$$

#### 第 3 步：用一阶线性 ODE 解掉 $\mathbf{x}_\lambda/\alpha_\lambda$

令
$$
\mathbf{u}(\lambda)=\frac{\mathbf{x}_\lambda}{\alpha_\lambda}
$$
由上式可得
$$
\frac{d\mathbf{u}}{d\lambda}+\mathbf{u}=\hat{\mathbf{x}}_\theta(\lambda)
$$

这是一阶线性 ODE，积分因子是 $e^{\lambda}$：
$$
\frac{d}{d\lambda}\left(e^{\lambda}\mathbf{u}(\lambda)\right)=e^{\lambda}\hat{\mathbf{x}}_\theta(\lambda)
$$
积分后：
$$
\mathbf{u}_t=e^{-(\lambda_t-\lambda_s)}\mathbf{u}_s+\int_{\lambda_s}^{\lambda_t}e^{-(\lambda_t-\lambda)}\hat{\mathbf{x}}_\theta(\lambda)d\lambda
$$
乘回 $\alpha_t$（且 $\mathbf{u}=\mathbf{x}/\alpha$）得到
$$
\boxed{
\mathbf{x}_t=\frac{\sigma_t}{\sigma_s}\mathbf{x}_s
+\alpha_t\int_{\lambda_s}^{\lambda_t}e^{-(\lambda_t-\lambda)}\hat{\mathbf{x}}_\theta(\lambda)d\lambda
}
$$

这就是 DPM-Solver++ 的 $x_0$ 形式。

#### 结论：你原式里第二个符号问题

你写的是 $e^{-(\lambda-\lambda_t)}=e^{\lambda_t-\lambda}$，指数方向反了。正确核应为
$$
e^{-(\lambda_t-\lambda)}=e^{\lambda-\lambda_t}
$$
并且前面是**正号积分项**（与官方实现完全一致）。

---

### 2. 二阶多步 DPM-Solver++（2M）详细讲解

我们要从 $\lambda_n$ 更新到 $\lambda_{n+1}$，并利用历史点 $\lambda_{n-1},\lambda_n$ 的模型值。

定义：
$$
h=\lambda_{n+1}-\lambda_n,\quad h_0=\lambda_n-\lambda_{n-1},\quad r_0=\frac{h_0}{h}
$$
$$
\hat{\mathbf{x}}_0^{(n)}=\hat{\mathbf{x}}_\theta(\lambda_n),\quad
\hat{\mathbf{x}}_0^{(n-1)}=\hat{\mathbf{x}}_\theta(\lambda_{n-1})
$$
$$
\mathbf{D}_1=\frac{1}{r_0}\left(\hat{\mathbf{x}}_0^{(n)}-\hat{\mathbf{x}}_0^{(n-1)}\right)
$$

并定义
$$
\phi_1(-h)=e^{-h}-1
$$

#### 2.1 一阶（用于启动）

若把区间内 $\hat{\mathbf{x}}_0(\lambda)$ 视为常数：
$$
\mathbf{x}_{n+1}=\frac{\sigma_{n+1}}{\sigma_n}\mathbf{x}_n-\alpha_{n+1}\phi_1(-h)\hat{\mathbf{x}}_0^{(n)}
$$

等价写法：
$$
\mathbf{x}_{n+1}=\frac{\sigma_{n+1}}{\sigma_n}\mathbf{x}_n+\alpha_{n+1}(1-e^{-h})\hat{\mathbf{x}}_0^{(n)}
$$

#### 2.2 二阶多步（官方 `solver_type="dpmsolver"`）

官方 PyTorch/JAX 实现里，二阶多步更新为：
$$
\boxed{
\mathbf{x}_{n+1}
=\frac{\sigma_{n+1}}{\sigma_n}\mathbf{x}_n
-\alpha_{n+1}\phi_1(-h)\hat{\mathbf{x}}_0^{(n)}
-\frac{1}{2}\alpha_{n+1}\phi_1(-h)\mathbf{D}_1
}
$$

这条式子对应代码中的 `multistep_dpm_solver_second_update`（`algorithm_type="dpmsolver++"`）。

#### 2.3 二阶多步（`solver_type="taylor"` 变体）

如果用线性插值做泰勒型推导，可得到：
$$
\mathbf{x}_{n+1}
=\frac{\sigma_{n+1}}{\sigma_n}\mathbf{x}_n
-\alpha_{n+1}\phi_1(-h)\hat{\mathbf{x}}_0^{(n)}
+\alpha_{n+1}\left(\frac{\phi_1(-h)}{h}+1\right)\mathbf{D}_1
$$

两者都是二阶；前者是仓库默认常见配置，后者是 Taylor 变体。

---

### 3. 二阶多步算法流程（实战）

1. 先用一阶式走第一步，拿到 $\hat{\mathbf{x}}_0^{(0)}$、$\hat{\mathbf{x}}_0^{(1)}$（用于“预热”多步法）。
2. 从第二个更新开始，每步保存最近两个 $\hat{\mathbf{x}}_0$。
3. 计算 $h,h_0,r_0,\mathbf{D}_1$。
4. 用上面的二阶多步公式更新 $\mathbf{x}_{n+1}$。
5. 滑动窗口：丢弃最旧点，保留最近两点进入下一步。

---

### 4. 为什么二阶多步更稳、更准

- 利用了时间方向上的“斜率信息” $\mathbf{D}_1$，不是只看当前点。
- 在同样网络调用次数下，局部截断误差从一阶提升到二阶。
- 对少步采样（如 10-20 步）尤其关键，通常比一阶更清晰、更稳定。